# Lettuce Color Quality Scoring Pipeline

## Objective

This notebook implements an automated pipeline for scoring lettuce images by **color quality**.
Each image (green or red variety) is assigned an ordinal quality score based on its color
distribution in the **CIELAB** perceptual color space. The pipeline is fully unsupervised.

**Dataset:** CSVS_Lettuce_202603_LaMenitre (color-corrected)
- **GREEN:** 21 images of green lettuce varieties
- **RED:** 17 images of red lettuce varieties

Images are 16-bit TIFF files captured under controlled greenhouse conditions.

## Pipeline Overview

| Step | Description |
|------|-------------|
| 1. Loading & Crop | Load 16-bit TIFFs, normalize to 8-bit RGB, extract inscribed-square crop |
| 2. Statistical Filter | Remove outlier pixels via +/-Kstd interval filter in CIELAB space |
| 3. Feature Extraction | Compute 27 statistical features per image (9 per LAB channel) |
| 4. Feature Engineering | Remove redundant features with \|correlation\| > 0.95 |
| 5. Quality Scoring | Cluster images into K ordinal scores (K=4 green, K=6 red) |
| 6. Export | Save scored images in structured folders + CSV metadata |

### Step 1 - Inscribed-Square Crop
The raw images contain a circular lettuce region on a dark background.
A **distance transform** identifies the largest inscribed circle, from which the largest
axis-aligned **inscribed square** is extracted. Pixels outside the circle are masked white.
This standardises the input and removes background artefacts.

### Step 2 - Statistical LAB Filter
Each crop is converted to **CIELAB** (perceptually uniform). For each channel (L\*, a\*, b\*),
only pixels within **[mean +/- 1.5std]** are retained. This removes specular highlights,
shadows, and non-representative colour artefacts.

### Step 3 - Feature Extraction
From each filtered image, **9 statistics per channel** are computed:
mean, std, P10, P25, P50 (median), P75, P90, skewness, kurtosis -> **27 raw features**.

### Step 4 - Feature Engineering
Features with pairwise |r| > 0.95 are removed to eliminate redundancy.
Channel means are protected from removal.

### Step 5 - Quality Scoring
Five complementary unsupervised methods assign ordinal scores:
- **A** - Weighted formula -> quantile bins
- **B** - K-Means clustering
- **C** - Hybrid (A-initialised K-Means)
- **D** - Equal-quantile a\* thresholding
- **E** - Adaptive Delta-E consecutive thresholding (K emerges from data)

Scores go from **1 (lightest)** to **K (darkest)**.
Within-cluster Delta-E CIEDE2000 validates colour homogeneity.

An **RGB channel projection** (R vs G) is also computed to verify whether
raw RGB provides discriminative power before committing to CIELAB.

### Step 6 - Export
Scored images saved to `<color>/Score=XX/` folders + master CSV.

In [ ]:
# ## 0 * Imports & Configuration
import os, glob, random, shutil
import numpy as np
import cv2
import tifffile as tiff
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.gridspec as mgridspec
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.stats import spearmanr, skew, kurtosis
from scipy.stats import kendalltau as ktau
from skimage.color import deltaE_ciede2000
from tqdm.auto import tqdm

plt.style.use('seaborn-v0_8')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# -- Paths ------------------------------------------------------------
LETTUCE_ROOT = r'C:\Users\aymen.masmoudi\Desktop\Database_CSVS\CSVS_Lettuce_202603_LaMenitre\Lettuce_COLOR_Evaluation_on_Greenhouse\OUTPUT\cc\CSVS01'
OUT_ROOT     = r'C:\Users\aymen.masmoudi\Desktop\stage\codes\pipeline_a_outputs\statistical_filtering_lettuce'
FIG_DIR      = os.path.join(OUT_ROOT, 'figures')

COLORS    = ['Score=GREEN', 'Score=RED']
COLOR_MAP = {'Score=GREEN': '#2ca02c', 'Score=RED': '#d62728'}

# -- Clustering K per color (Score 1 = lightest -> Score K = darkest) --
K_MAP = {'Score=GREEN': 4, 'Score=RED': 6}

# -- Delta-E threshold for within-cluster homogeneity -----------------
DE_THRESHOLD = 2.0    # max acceptable CIEDE2000 within a cluster
DE_THRESHOLD_E = {'Score=GREEN': 2.0, 'Score=RED': 3.0}  # per-color Method E threshold

# -- Filter params ----------------------------------------------------
K_STAT    = 1.5
FILTER_L  = True
FILTER_A  = True
FILTER_B  = True
FG_THRESH = 10

# -- Scoring axis (Score 1 = highest value = lightest) ----------------
SCORE_AXIS = 'L*_mean'

# -- Method A weights -------------------------------------------------
MA_W_L, MA_W_A, MA_W_B, MA_W_CHROMA = 1.0, 0.0, 0.0, 0.0

# -- Primary method for final export ----------------------------------
PRIMARY_METHOD = 'B'   # 'A', 'B', 'C', or 'D'

# -- Correlation cleaning threshold -----------------------------------
CORR_THRESHOLD = 0.95

# -- Create output folders --------------------------------------------
os.makedirs(FIG_DIR, exist_ok=True)
for color in COLORS:
    k = K_MAP[color]
    for s in range(1, k + 1):
        os.makedirs(os.path.join(OUT_ROOT, color, f'Score={s:02d}'), exist_ok=True)

print(f'Lettuce root  : {LETTUCE_ROOT}')
print(f'Output root   : {OUT_ROOT}')
print(f'K per color   : {K_MAP}')
print(f'DE threshold  : {DE_THRESHOLD}  (Method E: {DE_THRESHOLD_E})')
for c in COLORS:
    n = len(glob.glob(os.path.join(LETTUCE_ROOT, c, '*.tif')))
    print(f'  {c} -> {n} images')

## Step 1 - Image Loading & Inscribed-Square Crop

Each 16-bit TIFF image is normalised to 8-bit RGB, then an **inscribed-square crop** is extracted:
1. A foreground mask detects non-black pixels (any channel > 10).
2. The **distance transform** locates the centre and radius of the largest inscribed circle.
3. The largest axis-aligned square fitting inside that circle is cropped.
4. Pixels outside the circle (corners) are set to white.

This standardises all images to a uniform square region of the lettuce surface.

In [ ]:
def load_tif_uint8(path: str) -> np.ndarray:
    # Load uint16 TIFF -> normalise to uint8 RGB
    img = tiff.imread(path)
    if img.ndim == 4:
        img = img[0]
    if img.ndim == 3 and img.shape[0] in (1, 3, 4):
        img = np.transpose(img, (1, 2, 0))
    if img.shape[-1] == 4:
        img = img[..., :3]
    if img.dtype != np.uint8:
        lo, hi = img.min(), img.max()
        img = ((img.astype(np.float32) - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)
    return img


def get_inscribed_square_crop(img: np.ndarray) -> tuple:
    # Inscribed-square crop via distance transform
    H, W = img.shape[:2]
    fg_mask = np.any(img > FG_THRESH, axis=-1).astype(np.uint8)
    dist = cv2.distanceTransform(fg_mask, cv2.DIST_L2, 5)
    _, r_f, _, (cx, cy) = cv2.minMaxLoc(dist)
    r  = int(r_f)
    cx, cy = int(cx), int(cy)
    side = int(r * 2 ** 0.5)
    side = side - (side % 2)
    sx1 = max(0, cx - side // 2)
    sy1 = max(0, cy - side // 2)
    sx2 = min(W, sx1 + side)
    sy2 = min(H, sy1 + side)
    side = min(sx2 - sx1, sy2 - sy1)
    sx2, sy2 = sx1 + side, sy1 + side
    patch = img[sy1:sy2, sx1:sx2].copy()
    grid_y, grid_x = np.mgrid[sy1:sy2, sx1:sx2]
    outside = ((grid_x - cx) ** 2 + (grid_y - cy) ** 2) > r ** 2
    patch[outside] = 255
    info = {'cx': cx, 'cy': cy, 'r': r, 'side': side,
            'sx1': sx1, 'sy1': sy1, 'sx2': sx2, 'sy2': sy2}
    return patch, info


# -- Load and crop all images -----------------------------------------
dataset = {}

for color in COLORS:
    folder  = os.path.join(LETTUCE_ROOT, color)
    files   = sorted(glob.glob(os.path.join(folder, '*.tif')))
    entries = []
    for fpath in files:
        raw = load_tif_uint8(fpath)
        cropped, info = get_inscribed_square_crop(raw)
        entries.append({'path': fpath, 'raw': raw, 'cropped': cropped,
                        'crop_info': info, 'fname': os.path.basename(fpath)})
    dataset[color] = entries
    print(f'{color}: {len(entries)} images loaded - crop shape {entries[0]["cropped"].shape}')

print(f'Total: {sum(len(v) for v in dataset.values())} images')

## Step 2 - Crop Visualisation

One sample per colour: original image with inscribed circle overlay, kept-vs-discarded map, and final crop.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle('Inscribed-Square Crop - One Sample per Colour', fontsize=13, fontweight='bold')

for row_i, color in enumerate(COLORS):
    entry = dataset[color][0]
    raw, cropped = entry['raw'], entry['cropped']
    info = entry['crop_info']
    cx, cy, r = info['cx'], info['cy'], info['r']
    sx1, sy1, sx2, sy2 = info['sx1'], info['sy1'], info['sx2'], info['sy2']
    side = info['side']
    col = COLOR_MAP[color]

    axes[row_i, 0].imshow(raw)
    circ = plt.Circle((cx, cy), r, fill=False, color=col, linewidth=2, linestyle='--')
    rect = mpatches.Rectangle((sx1, sy1), side, side,
                               linewidth=2, edgecolor='gold', facecolor='none')
    axes[row_i, 0].add_patch(circ)
    axes[row_i, 0].add_patch(rect)
    axes[row_i, 0].set_title(f'{color}\nOriginal {raw.shape[1]}x{raw.shape[0]}  r={r}  side={side}',
                              fontsize=9, color=col, fontweight='bold')
    axes[row_i, 0].axis('off')

    discard = raw.copy()
    H, W = raw.shape[:2]
    gy, gx = np.mgrid[0:H, 0:W]
    outside = ((gx - cx)**2 + (gy - cy)**2) > r**2
    discard[outside] = 200
    axes[row_i, 1].imshow(discard)
    axes[row_i, 1].set_title('Kept (colour) vs Discarded (gray)', fontsize=9)
    axes[row_i, 1].axis('off')

    axes[row_i, 2].imshow(cropped)
    axes[row_i, 2].set_title(f'Cropped {side}x{side}', fontsize=9, color=col, fontweight='bold')
    axes[row_i, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'crop_visualisation.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved crop_visualisation.png')

## Step 3 - Statistical LAB Filter

Each cropped image is converted to **CIELAB** colour space.  For each channel (L\*, a\*, b\*),
foreground pixels are retained only if their value lies within
**[mean - K*std,  mean + K*std]** (K = 1.5 by default).

This removes outlier pixels - specular highlights, deep shadows, and non-lettuce artefacts -
while preserving the representative colour distribution.

In [ ]:
def stat_filter(img_rgb: np.ndarray, k: float = 1.5,
                filter_l: bool = True, filter_a: bool = True,
                filter_b: bool = True) -> dict:
    # Statistical LAB interval filter on foreground pixels
    H, W = img_rgb.shape[:2]
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    lab   = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[..., 0] * 100.0 / 255.0
    a_img = lab[..., 1] - 128.0
    b_img = lab[..., 2] - 128.0

    coords = np.argwhere(fg_mask)
    if len(coords) == 0:
        return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0,
                'stats': {}, 'fg_vals': {}, 'kept_px': np.array([], dtype=bool)}

    r, c = coords[:, 0], coords[:, 1]
    channels = {'L*': (L_img, filter_l), 'a*': (a_img, filter_a), 'b*': (b_img, filter_b)}
    keep_px = np.ones(len(coords), dtype=bool)
    stats_out, fg_vals = {}, {}

    for ch_name, (ch_img, active) in channels.items():
        vals = ch_img[r, c]
        med  = float(np.mean(vals))
        std  = float(np.std(vals))
        lo, hi = med - k * std, med + k * std
        stats_out[ch_name] = {'mean': med, 'std': std, 'lo': lo, 'hi': hi, 'active': active}
        fg_vals[ch_name] = vals
        if active:
            keep_px &= (vals >= lo) & (vals <= hi)

    kept_mask = np.zeros((H, W), bool)
    kept_mask[r[keep_px], c[keep_px]] = True
    out = np.full_like(img_rgb, 255)
    out[kept_mask] = img_rgb[kept_mask]

    return {'out': out, 'kept_mask': kept_mask,
            'kept_pct': float(keep_px.sum() / len(coords) * 100),
            'stats': stats_out, 'fg_vals': fg_vals, 'kept_px': keep_px}


# -- Apply to all images ----------------------------------------------
for color in COLORS:
    for entry in dataset[color]:
        res = stat_filter(entry['cropped'], k=K_STAT,
                          filter_l=FILTER_L, filter_a=FILTER_A, filter_b=FILTER_B)
        entry.update(res)
    kept_pcts = [e['kept_pct'] for e in dataset[color]]
    print(f'{color}: kept% = {np.mean(kept_pcts):.1f}% +/- {np.std(kept_pcts):.1f}% '
          f'(min={np.min(kept_pcts):.1f}%  max={np.max(kept_pcts):.1f}%)')

print(f'\nFilter: K = +/-{K_STAT}s  |  L*={FILTER_L}  a*={FILTER_A}  b*={FILTER_B}')

## Step 4 - Filter Results (Representative Samples)

For each colour, **4 representative images** are shown with their original crop, filtered output,
and per-channel histograms highlighting the kept (dark) vs rejected (light) pixel distributions.
The red dashed lines mark the +/-Kstd filter boundaries.

In [ ]:
CHANNEL_COLORS = {'L*': 'gray', 'a*': 'green', 'b*': 'goldenrod'}
CHANNEL_UNITS  = {'L*': '[0,100]', 'a*': '[-127,127]', 'b*': '[-127,127]'}
N_VIZ_SAMPLES  = 4   # images per colour

for color in COLORS:
    entries = dataset[color]
    n = len(entries)
    indices = np.linspace(0, n - 1, N_VIZ_SAMPLES, dtype=int)

    for idx in indices:
        entry   = entries[idx]
        stats   = entry['stats']
        fg_vals = entry['fg_vals']
        kept_px = entry['kept_px']
        if len(kept_px) == 0:
            continue

        active_chs = [ch for ch, s in stats.items() if s['active']]
        n_rows = 1 + len(active_chs)

        fig = plt.figure(figsize=(13, 3 + len(active_chs) * 3))
        gs  = GridSpec(n_rows, 2, figure=fig, hspace=0.45, wspace=0.3)
        src_col = COLOR_MAP[color]

        ax0 = fig.add_subplot(gs[0, 0])
        ax1 = fig.add_subplot(gs[0, 1])
        ax0.imshow(entry['cropped'])
        ax0.set_title(f'{color} | {entry["fname"]}\nCropped (original)',
                      fontsize=9, color=src_col, fontweight='bold')
        ax0.axis('off')
        ax1.imshow(entry['out'])
        ax1.set_title(f'Filtered  kept={entry["kept_pct"]:.1f}%', fontsize=9)
        ax1.axis('off')

        for ri, ch in enumerate(active_chs):
            s    = stats[ch]
            vals = fg_vals[ch]
            col  = CHANNEL_COLORS[ch]
            ax   = fig.add_subplot(gs[1 + ri, :])
            ax.hist(vals, bins=120, color=col, alpha=0.3, label='All fg')
            ax.hist(vals[kept_px], bins=120, color=col, alpha=0.75, label='Kept')
            ax.axvline(s['mean'], color='black', lw=2, ls='-',
                       label=f"mean={s['mean']:.2f}")
            ax.axvline(s['lo'], color='crimson', lw=1.5, ls='--',
                       label=f"lo={s['lo']:.2f}")
            ax.axvline(s['hi'], color='crimson', lw=1.5, ls='--',
                       label=f"hi={s['hi']:.2f}")
            ax.axvspan(s['lo'], s['hi'], alpha=0.1, color=col)
            ax.set_xlabel(f'{ch}  {CHANNEL_UNITS[ch]}')
            ax.set_ylabel('Pixel count')
            ax.set_title(f'{ch} - mean={s["mean"]:.2f}  std={s["std"]:.2f}  '
                         f'interval=[{s["lo"]:.2f}, {s["hi"]:.2f}]')
            ax.legend(fontsize=8, ncol=4); ax.grid(alpha=0.3)

        safe = entry['fname'].replace('.tif', '')
        fig.suptitle(f'Statistical Filter | {color} | {entry["fname"]} | K=+/-{K_STAT}s',
                     fontsize=10, fontweight='bold', y=1.01)
        save_dir = os.path.join(OUT_ROOT, 'filter_viz', color)
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, f'{safe}_filter.png'),
                    dpi=100, bbox_inches='tight')
        plt.show(); plt.close()
        print(f'  [{color}] {entry["fname"]} - kept={entry["kept_pct"]:.1f}%')

## Step 5 - LAB Feature Extraction

From each filtered image, **9 statistics** are computed per CIELAB channel:

| Statistic | Description |
|-----------|-------------|
| Mean | Average channel value across filtered pixels |
| Std | Standard deviation - colour variability |
| P10, P25, P50, P75, P90 | Percentile profile - captures the full distribution shape |
| Skewness | Asymmetry of the distribution (0 = symmetric) |
| Kurtosis | Tailedness - peaky (high) vs flat (low) distribution |

**Total:** 9 features x 3 channels (L\*, a\*, b\*) = **27 raw features** per image.

In [ ]:
def extract_lab_features(img_rgb: np.ndarray, fg_mask: np.ndarray) -> np.ndarray:
    # 27-dim LAB feature vector from filtered foreground pixels
    lab   = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[..., 0] * 100.0 / 255.0
    a_img = lab[..., 1] - 128.0
    b_img = lab[..., 2] - 128.0

    feats = []
    for ch in [L_img, a_img, b_img]:
        vals = ch[fg_mask]
        feats.append(vals.mean())
        feats.append(vals.std())
        feats.extend(np.percentile(vals, [10, 25, 50, 75, 90]).tolist())
        feats.append(float(skew(vals)))
        feats.append(float(kurtosis(vals)))
    return np.array(feats, dtype=np.float32)


# Feature names (27 total)
feat_names_raw = []
for ch in ['L*', 'a*', 'b*']:
    feat_names_raw += [f'{ch}_mean', f'{ch}_std',
                       f'{ch}_P10', f'{ch}_P25', f'{ch}_P50',
                       f'{ch}_P75', f'{ch}_P90',
                       f'{ch}_skew', f'{ch}_kurt']

# Extract features for all images
for color in COLORS:
    for entry in dataset[color]:
        entry['features'] = extract_lab_features(entry['out'], entry['kept_mask'])
    print(f'{color}: features extracted - shape {dataset[color][0]["features"].shape}')

# Build raw feature matrices
feat_matrices_raw = {}
for color in COLORS:
    X = np.stack([e['features'] for e in dataset[color]])
    feat_matrices_raw[color] = X
    print(f'{color}: feature matrix {X.shape}')

# -- Display feature tables per channel --------------------------------
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

for ch in ['L*', 'a*', 'b*']:
    ch_feats = [n for n in feat_names_raw if n.startswith(ch)]
    ch_idx   = [feat_names_raw.index(n) for n in ch_feats]
    rows = []
    for color in COLORS:
        X = feat_matrices_raw[color]
        for i, e in enumerate(dataset[color]):
            row = {'Color': color.replace('Score=', ''),
                   'Image': e['fname'].replace('CSVS01_', '').replace('.tif', '')}
            for fi, fn in zip(ch_idx, ch_feats):
                row[fn.replace(f'{ch}_', '')] = round(float(X[i, fi]), 2)
            rows.append(row)
    df = pd.DataFrame(rows)
    print(f'\n{"="*80}')
    print(f'  {ch} Channel Features ({len(ch_feats)} features x {len(rows)} images)')
    print(f'{"="*80}')
    print(df.to_string(index=False))

## Step 6 - Feature Engineering: Correlation-Based Cleaning

Highly correlated features carry redundant information and can degrade clustering quality.
We remove features whose pairwise **Pearson |r| > 0.95**, keeping the one with lower average
correlation to all other features. The **channel means** (L\*\_mean, a\*\_mean, b\*\_mean)
are always protected from removal.

This produces a compact, non-redundant feature set for downstream clustering.

In [ ]:
PROTECT_FEATURES = ['L*_mean', 'a*_mean', 'b*_mean']

def remove_correlated(X, names, threshold=0.95, protect=None):
    # Remove features with |correlation| > threshold
    # Protected features are never dropped
    protect = set(protect or [])
    corr = np.abs(np.corrcoef(X.T))
    n = corr.shape[0]
    avg_corr = corr.mean(axis=1)
    drop = set()
    for i in range(n):
        if i in drop:
            continue
        for j in range(i + 1, n):
            if j in drop:
                continue
            if corr[i, j] > threshold:
                if names[j] in protect:
                    if names[i] not in protect:
                        drop.add(i)
                elif names[i] in protect:
                    drop.add(j)
                elif avg_corr[j] >= avg_corr[i]:
                    drop.add(j)
                else:
                    drop.add(i)
    keep = sorted(set(range(n)) - drop)
    return X[:, keep], [names[i] for i in keep], keep


# Combine all images for consistent cleaning
X_all_raw = np.vstack([feat_matrices_raw[c] for c in COLORS])
X_all_clean, feat_names, keep_idx = remove_correlated(
    X_all_raw, feat_names_raw, threshold=CORR_THRESHOLD, protect=PROTECT_FEATURES)

# Split back per colour
feat_matrices = {}
offset = 0
for color in COLORS:
    n = len(dataset[color])
    feat_matrices[color] = X_all_clean[offset:offset + n]
    offset += n

# Report
removed = [n for i, n in enumerate(feat_names_raw) if i not in keep_idx]
print(f'Raw features  : {len(feat_names_raw)}')
print(f'After cleaning: {len(feat_names)}  (threshold={CORR_THRESHOLD})')
print(f'Removed ({len(removed)}): {removed}')
print(f'Kept    ({len(feat_names)}): {feat_names}')

# Show correlation matrix of cleaned features
fig, ax = plt.subplots(figsize=(10, 8))
corr_clean = np.corrcoef(X_all_clean.T)
im = ax.imshow(corr_clean, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(feat_names)))
ax.set_xticklabels(feat_names, rotation=90, fontsize=7)
ax.set_yticks(range(len(feat_names)))
ax.set_yticklabels(feat_names, fontsize=7)
for i in range(len(feat_names)):
    for j in range(len(feat_names)):
        ax.text(j, i, f'{corr_clean[i,j]:.2f}', ha='center', va='center', fontsize=5)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Pearson r')
ax.set_title(f'Feature Correlation Matrix (after cleaning, {len(feat_names)} features)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'correlation_matrix_cleaned.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved correlation_matrix_cleaned.png')

## Step 7 - Statistical Feature Visualisations

Four complementary plots explore the cleaned feature space:

1. **Sorted bar plots** - For each key feature (means, stds), images are sorted by value.
   This reveals the natural ordering and separability between colours.
2. **Feature heatmap** - All cleaned features x all images, min-max normalised.
   Highlights patterns across the full feature space.
3. **Boxplots** - GREEN vs RED distribution comparison for key features.
   Shows inter-colour separability and within-colour variability.
4. **Scatter matrix** - L\*, a\*, b\* means + Chroma pairwise relationships.
   Reveals clustering structure in the most discriminative feature subspace.

In [ ]:
# -- 1. Sorted Bar Plots - key features sorted by value ---------------
# Images sorted by feature value -> reveals natural ordering and colour separation

KEY_FEATS = ['L*_mean', 'L*_std', 'a*_mean', 'a*_std', 'b*_mean', 'b*_std']
KEY_FEATS_AVAIL = [f for f in KEY_FEATS if f in feat_names]
n_kf = len(KEY_FEATS_AVAIL)

fig, axes = plt.subplots(2, (n_kf + 1) // 2, figsize=(5.5 * ((n_kf + 1) // 2), 10))
axes = axes.ravel()
fig.suptitle('Key LAB Features - Sorted by Value (all images)',
             fontsize=13, fontweight='bold')

for ci, feat in enumerate(KEY_FEATS_AVAIL):
    ax = axes[ci]
    fi = feat_names.index(feat)

    # Collect all images with labels
    all_vals, all_cols, all_labels = [], [], []
    for color in COLORS:
        X = feat_matrices[color]
        for i, e in enumerate(dataset[color]):
            all_vals.append(float(X[i, fi]))
            all_cols.append(COLOR_MAP[color])
            all_labels.append(e['fname'].replace('CSVS01_', '').replace('.tif', ''))

    # Sort by value
    order = np.argsort(all_vals)
    sorted_vals   = [all_vals[i] for i in order]
    sorted_cols   = [all_cols[i] for i in order]
    sorted_labels = [all_labels[i] for i in order]

    ax.bar(range(len(sorted_vals)), sorted_vals, color=sorted_cols,
           alpha=0.75, edgecolor='white', linewidth=0.5)
    mean_val = np.mean(sorted_vals)
    ax.axhline(mean_val, color='black', lw=1.5, ls='--',
               label=f'mean={mean_val:.2f}')
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(sorted_labels)))
    ax.set_xticklabels(sorted_labels, rotation=90, fontsize=5)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3, axis='y')

# Hide unused axes
for ci in range(n_kf, len(axes)):
    axes[ci].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_barplots_sorted.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved feature_barplots_sorted.png')

In [ ]:
# -- 2. Feature Heatmap - all cleaned features x all images ------------
# Reveals per-image patterns across the full feature space

fig, axes = plt.subplots(1, len(COLORS), figsize=(7 * len(COLORS), 10))
fig.suptitle(f'Feature Heatmap - {len(feat_names)} Cleaned Features (min-max normalised)',
             fontsize=12, fontweight='bold')

for ci, color in enumerate(COLORS):
    X      = feat_matrices[color]
    X_norm = MinMaxScaler().fit_transform(X.T).T
    ax     = axes[ci] if len(COLORS) > 1 else axes
    fnames = [e['fname'].replace('CSVS01_', '').replace('.tif', '')
              for e in dataset[color]]

    im = ax.imshow(X_norm.T, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_yticks(range(len(feat_names)))
    ax.set_yticklabels(feat_names, fontsize=6)
    ax.set_xticks(range(len(fnames)))
    ax.set_xticklabels(fnames, rotation=90, fontsize=6)
    ax.set_title(color, fontsize=10, fontweight='bold', color=COLOR_MAP[color])
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_heatmap.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved feature_heatmap.png')

In [ ]:
# -- 3. Boxplots - GREEN vs RED comparison for key features ------------
# Shows inter-colour separation and within-colour spread

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Key Feature Distributions - GREEN vs RED', fontsize=12, fontweight='bold')

for idx, feat in enumerate(KEY_FEATS_AVAIL[:6]):
    ax = axes[idx // 3, idx % 3]
    fi = feat_names.index(feat) if feat in feat_names else None
    if fi is None:
        ax.set_visible(False); continue
    data = [feat_matrices[c][:, fi] for c in COLORS]
    cols = [COLOR_MAP[c] for c in COLORS]
    bp = ax.boxplot(data, labels=[c.replace('Score=', '') for c in COLORS],
                    patch_artist=True, widths=0.4)
    for patch, c in zip(bp['boxes'], cols):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    for ele in ['whiskers', 'caps', 'medians', 'fliers']:
        for item in bp[ele]:
            item.set(color='black', linewidth=1.2)
    for xi, (d, c) in enumerate(zip(data, cols), start=1):
        ax.scatter(np.random.normal(xi, 0.05, len(d)), d,
                   color=c, alpha=0.8, s=30, zorder=5,
                   edgecolors='white', linewidths=0.3)
    ax.set_title(feat, fontsize=9)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_boxplots.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved feature_boxplots.png')

In [ ]:
# -- 4. Scatter Matrix - L*, a*, b* means + Chroma --------------------
# Reveals the 2D clustering structure in the most discriminative subspace

_Li = feat_names.index('L*_mean')
_Ai = feat_names.index('a*_mean')
_Bi = feat_names.index('b*_mean')

rows_sc = []
for color in COLORS:
    X = feat_matrices[color]
    for i, e in enumerate(dataset[color]):
        rows_sc.append({
            'color': color,
            'L* mean': X[i, _Li], 'a* mean': X[i, _Ai], 'b* mean': X[i, _Bi],
            'Chroma': np.sqrt(X[i, _Ai]**2 + X[i, _Bi]**2)
        })
df_sc = pd.DataFrame(rows_sc)

scatter_keys = ['L* mean', 'a* mean', 'b* mean', 'Chroma']
n_s = len(scatter_keys)
fig, axes = plt.subplots(n_s, n_s, figsize=(3.5 * n_s, 3.5 * n_s))
fig.suptitle('Scatter Matrix - L*/a*/b* Means + Chroma', fontsize=12, fontweight='bold')

for ri, ky in enumerate(scatter_keys):
    for ci, kx in enumerate(scatter_keys):
        ax = axes[ri, ci]
        if ri == ci:
            for color in COLORS:
                sub = df_sc[df_sc['color'] == color]
                ax.hist(sub[ky], bins=10, color=COLOR_MAP[color], alpha=0.6, label=color)
            ax.set_title(ky, fontsize=8)
            if ri == 0:
                ax.legend(fontsize=6)
        else:
            for color in COLORS:
                sub = df_sc[df_sc['color'] == color]
                ax.scatter(sub[kx], sub[ky], c=COLOR_MAP[color], alpha=0.7,
                           s=60, edgecolors='white', linewidths=0.4)
        ax.grid(alpha=0.3)
        if ci == 0: ax.set_ylabel(ky, fontsize=7)
        if ri == n_s - 1: ax.set_xlabel(kx, fontsize=7)
        ax.tick_params(labelsize=6)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_scatter_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved feature_scatter_matrix.png')

## RGB Channel Projection - Red vs Green

Before committing to CIELAB, we check whether the **R and G channels** alone
provide sufficient discriminative power between GREEN and RED lettuce varieties.
For each image, the mean R and G values are computed from filtered foreground pixels.

In [ ]:
# --- RGB R/G Channel Projection ---
rgb_rows = []
for color in COLORS:
    for entry in dataset[color]:
        fg_pixels = entry['out'][entry['kept_mask']]  # (N, 3)
        mean_r = float(fg_pixels[:, 0].mean())
        mean_g = float(fg_pixels[:, 1].mean())
        mean_b = float(fg_pixels[:, 2].mean())
        rgb_rows.append({
            'color': color, 'fname': entry['fname'],
            'R_mean': mean_r, 'G_mean': mean_g, 'B_mean': mean_b
        })
        entry['rgb_mean'] = np.array([mean_r, mean_g, mean_b])

df_rgb = pd.DataFrame(rgb_rows)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RGB Channel Projections (filtered foreground pixels)',
             fontsize=13, fontweight='bold')

pairs = [('R_mean', 'G_mean'), ('R_mean', 'B_mean'), ('G_mean', 'B_mean')]
for ax, (xk, yk) in zip(axes, pairs):
    for color in COLORS:
        sub = df_rgb[df_rgb['color'] == color]
        ax.scatter(sub[xk], sub[yk], c=COLOR_MAP[color], s=80,
                   edgecolors='white', linewidths=0.5, alpha=0.8,
                   label=color.replace('Score=', ''))
    ax.set_xlabel(xk); ax.set_ylabel(yk)
    ax.set_title(f'{xk} vs {yk}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'rgb_projection.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved rgb_projection.png')

# --- Sorted bar: mean R and G per image ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mean R and G Channel Values (sorted)', fontsize=12, fontweight='bold')

for ai, ch_name in enumerate(['R_mean', 'G_mean']):
    ax = axes[ai]
    vals = df_rgb[ch_name].values
    colors_list = [COLOR_MAP[c] for c in df_rgb['color']]
    labels = [r.replace('CSVS01_', '').replace('.tif', '') for r in df_rgb['fname']]
    order = np.argsort(vals)
    ax.bar(range(len(vals)), vals[order],
           color=[colors_list[i] for i in order], alpha=0.75)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([labels[i] for i in order], rotation=90, fontsize=5)
    ax.set_title(ch_name); ax.set_ylabel('Pixel value (0-255)')
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'rgb_sorted_bars.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved rgb_sorted_bars.png')

## Step 8 - Colour Quality Scoring via Clustering

Each colour group is clustered **independently** into ordinal quality scores:
- **GREEN:** K = 4 clusters
- **RED:** K = 6 clusters

**Score convention:** Score 1 = lightest (highest L\*) -> Score K = darkest (lowest L\*).

### Method A - Weighted Formula -> Quantile Bins
A scalar quality index Q = w_L*(L\*/100) + w_a*a\* + w_b*b\* + w_chroma*C\*
is computed. Images are ranked by Q (descending) and split into K equal-quantile bins.
*Pros:* deterministic, physically grounded. *Cons:* requires weight tuning.

### Method B - K-Means Clustering
K-Means clusters the full cleaned feature space. Cluster centroids are sorted by L\* mean
(descending), so Score 1 = lightest centroid. *Pros:* data-driven, no weight tuning.

### Method C - Hybrid (A-initialised K-Means)
Method A bins provide the initial centroids for K-Means, combining physical grounding
with natural data boundaries. *Pros:* best of both worlds.

### Method D - a\* Thresholding
Equal-quantile cuts on `a*_mean`, then bins relabelled so Score 1 = lightest L\*.
*Pros:* fully interpretable, single-feature. *Cons:* ignores other features.

### Method E - Adaptive Delta-E Consecutive Thresholding
Images are sorted by L\* (lightest first) and Delta-E CIEDE2000 is computed between each
consecutive pair. A cluster boundary is placed wherever Delta-E exceeds a threshold.
The number of clusters K **emerges naturally** from the data rather than being fixed.
*Pros:* no K selection needed, perceptually grounded boundaries. *Cons:* sensitive to threshold.

### Validation - Within-cluster Delta-E CIEDE2000
For each cluster, the maximum and mean CIEDE2000 colour difference between all image pairs
is computed. Clusters with max Delta-E > threshold indicate heterogeneous colour groupings.

In [ ]:
# -- Feature indices (from cleaned feature set) -----------------------
_L = feat_names.index('L*_mean')
_A = feat_names.index('a*_mean')
_B = feat_names.index('b*_mean')


def _order_centres(centres):
    # Sort cluster centres by SCORE_AXIS descending (lightest = Score 1)
    idx = feat_names.index(SCORE_AXIS)
    return np.argsort(-centres[:, idx])


# -- Method A - weighted formula -> quantile bins ---------------------
def method_a_scores(X, k):
    l      = X[:, _L] / 100.0
    a      = X[:, _A]
    b      = X[:, _B]
    chroma = np.sqrt(a**2 + b**2)
    Q      = MA_W_L * l + MA_W_A * a + MA_W_B * b + MA_W_CHROMA * chroma
    n      = len(Q)
    rk     = np.argsort(np.argsort(-Q))
    sc     = (np.floor(rk / n * k).astype(int) + 1).clip(1, k)
    return sc.astype(int), Q


# -- Method B - K-Means -----------------------------------------------
def method_b_scores(X, k):
    sc_   = StandardScaler()
    Xs    = sc_.fit_transform(X)
    km    = KMeans(n_clusters=k, random_state=SEED, n_init=20, max_iter=500)
    lbls  = km.fit_predict(Xs)
    ctrs  = sc_.inverse_transform(km.cluster_centers_)
    order = _order_centres(ctrs)
    sc_map = {old: new + 1 for new, old in enumerate(order)}
    scores = np.array([sc_map[lb] for lb in lbls])
    sil    = silhouette_score(Xs, lbls) if len(set(lbls)) > 1 else 0.0
    return scores.astype(int), sil, ctrs


# -- Method C - Hybrid (A init -> K-Means) ----------------------------
def method_c_scores(X, k):
    sc_   = StandardScaler()
    Xs    = sc_.fit_transform(X)
    a_sc, _ = method_a_scores(X, k)
    init  = np.stack([
        Xs[a_sc == s].mean(axis=0) if (a_sc == s).any() else Xs.mean(axis=0)
        for s in range(1, k + 1)])
    km    = KMeans(n_clusters=k, init=init, n_init=1,
                   max_iter=500, random_state=SEED)
    lbls  = km.fit_predict(Xs)
    ctrs  = sc_.inverse_transform(km.cluster_centers_)
    order = _order_centres(ctrs)
    sc_map = {old: new + 1 for new, old in enumerate(order)}
    scores = np.array([sc_map[lb] for lb in lbls])
    sil    = silhouette_score(Xs, lbls) if len(set(lbls)) > 1 else 0.0
    return scores.astype(int), sil


# -- Method D - a* thresholding ---------------------------------------
def method_d_scores(X, k, feature='a*_mean'):
    fi   = feat_names.index(feature)
    vals = X[:, fi]
    pcts = np.linspace(0, 100, k + 1)
    edges = list(np.percentile(vals, pcts))
    edges[0] = -np.inf; edges[-1] = np.inf
    raw = np.zeros(len(vals), dtype=int)
    for i in range(k):
        mask = (vals > edges[i]) & (vals <= edges[i + 1])
        raw[mask] = i + 1
    raw = raw.clip(1, k)
    ax_fi   = feat_names.index(SCORE_AXIS)
    ax_vals = X[:, ax_fi]
    bin_means = np.array([
        ax_vals[raw == b].mean() if (raw == b).any() else -np.inf
        for b in range(1, k + 1)])
    order  = np.argsort(-bin_means)
    sc_map = {old + 1: new + 1 for new, old in enumerate(order)}
    scores = np.array([sc_map[b] for b in raw])
    return scores.astype(int)


# -- Delta-E within-cluster validation ---------------------------------
def validate_cluster_deltaE(X, scores, k):
    # Compute within-cluster CIEDE2000 colour distances
    results = {}
    for s in range(1, k + 1):
        mask = scores == s
        n = int(mask.sum())
        if n < 2:
            results[s] = {'n': n, 'max_dE': 0.0, 'mean_dE': 0.0, 'ok': True}
            continue
        labs = X[mask][:, [_L, _A, _B]]
        des = []
        for i in range(len(labs)):
            for j in range(i + 1, len(labs)):
                lab1 = labs[i].reshape(1, 1, 3)
                lab2 = labs[j].reshape(1, 1, 3)
                de = float(deltaE_ciede2000(lab1, lab2)[0, 0])
                des.append(de)
        results[s] = {
            'n': n,
            'max_dE': round(max(des), 2),
            'mean_dE': round(np.mean(des), 2),
            'ok': max(des) <= DE_THRESHOLD
        }
    return results


print('Scoring method functions defined.')

## Method E - Adaptive Delta-E Consecutive Thresholding

A new scoring method where K is **not fixed** but emerges from the data:
1. Compute the mean LAB (L*, a*, b*) per image from filtered foreground
2. Sort images by L* (lightest first)
3. Compute Delta-E CIEDE2000 between each **consecutive pair** in sorted order
4. If Delta-E between two neighbours > threshold, place a **cluster boundary**
5. K emerges naturally from the data

Multiple thresholds (1.0, 1.5, 2.0, 2.5, 3.0) are tested.

In [ ]:
# --- Method E: Adaptive Delta-E consecutive thresholding ---
def method_e_scores(X, de_threshold=2.0):
    # Sort by L* descending (lightest=Score 1), split where consecutive dE > threshold
    n = len(X)
    labs = X[:, [_L, _A, _B]]
    sort_idx = np.argsort(-labs[:, 0])
    sorted_labs = labs[sort_idx]

    # Consecutive Delta-E
    consec_de = []
    for i in range(n - 1):
        lab1 = sorted_labs[i].reshape(1, 1, 3)
        lab2 = sorted_labs[i + 1].reshape(1, 1, 3)
        de = float(deltaE_ciede2000(lab1, lab2)[0, 0])
        consec_de.append(de)

    # Cluster assignment: new cluster when dE > threshold
    cluster = 1
    sorted_scores = np.ones(n, dtype=int)
    for i, de in enumerate(consec_de):
        if de > de_threshold:
            cluster += 1
        sorted_scores[i + 1] = cluster

    # Map back to original order
    scores = np.zeros(n, dtype=int)
    for rank, orig_idx in enumerate(sort_idx):
        scores[orig_idx] = sorted_scores[rank]

    return scores.astype(int), int(max(scores)), np.array(consec_de), sort_idx

print('Method E (adaptive Delta-E thresholding) defined.')

In [ ]:
# --- Threshold sweep for Method E ---
DE_THRESHOLDS = [1.0, 1.5, 2.0, 2.5, 3.0]

for color in COLORS:
    X = feat_matrices[color]
    entries = dataset[color]
    n_img = len(entries)
    col_lbl = color.replace('Score=', '')
    src_col = COLOR_MAP[color]

    # Compute consecutive Delta-E (same sorted order for all thresholds)
    labs = X[:, [_L, _A, _B]]
    sort_idx = np.argsort(-labs[:, 0])
    sorted_labs = labs[sort_idx]
    consec_de = []
    for i in range(n_img - 1):
        lab1 = sorted_labs[i].reshape(1, 1, 3)
        lab2 = sorted_labs[i + 1].reshape(1, 1, 3)
        consec_de.append(float(deltaE_ciede2000(lab1, lab2)[0, 0]))

    # --- Bar chart: consecutive Delta-E with threshold lines ---
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(range(len(consec_de)), consec_de, color=src_col, alpha=0.7, edgecolor='white')
    for th in DE_THRESHOLDS:
        ax.axhline(th, linestyle='--', linewidth=1, alpha=0.7, label=f'th={th}')
    sorted_names = [entries[i]['fname'].replace('CSVS01_', '').replace('.tif', '')
                    for i in sort_idx]
    pair_labels = [f'{sorted_names[i]}>{sorted_names[i+1]}'
                   for i in range(len(consec_de))]
    ax.set_xticks(range(len(consec_de)))
    ax.set_xticklabels(pair_labels, fontsize=4, rotation=90)
    ax.set_ylabel('Delta-E CIEDE2000')
    ax.set_title(f'{col_lbl} - Consecutive Delta-E (sorted by L*)',
                 fontsize=11, fontweight='bold', color=src_col)
    ax.legend(fontsize=7, ncol=len(DE_THRESHOLDS))
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'deltae_consecutive_{col_lbl.lower()}.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved deltae_consecutive_{col_lbl.lower()}.png')

    # Summary table
    print(f'{col_lbl} - Method E threshold sweep:')
    print(f'  {"Threshold":<12} {"K":<5} {"Distribution"}')
    for th in DE_THRESHOLDS:
        scores_e, k_e, _, _ = method_e_scores(X, de_threshold=th)
        dist = {s: int((scores_e == s).sum()) for s in range(1, k_e + 1)}
        print(f'  {th:<12} {k_e:<5} {dist}')


In [ ]:
# Compute all five methods per colour + Delta-E validation + save images

method_results = {}

for color in COLORS:
    X    = feat_matrices[color]
    ents = dataset[color]
    k    = K_MAP[color]

    print(f'\n{"="*65}')
    print(f'  {color}  ({X.shape[0]} images, K={k})')
    print(f'{"="*65}')

    a_sc, a_Q      = method_a_scores(X, k)
    b_sc, b_sil, _ = method_b_scores(X, k)
    c_sc, c_sil    = method_c_scores(X, k)
    d_sc           = method_d_scores(X, k)
    de_th_e = DE_THRESHOLD_E.get(color, DE_THRESHOLD)
    e_sc, k_e, _, _ = method_e_scores(X, de_threshold=de_th_e)

    print(f'  Silhouette:  B={b_sil:.4f}  C={c_sil:.4f}')
    print(f'  Method E: K_adaptive={k_e} (threshold={de_th_e})')

    method_results[color] = {
        'A': a_sc, 'B': b_sc, 'C': c_sc, 'D': d_sc, 'E': e_sc,
        'b_sil': b_sil, 'c_sil': c_sil, 'k': k, 'k_e': k_e
    }

    # Attach scores to dataset entries
    for i, e in enumerate(ents):
        e['score_A'] = int(a_sc[i])
        e['score_B'] = int(b_sc[i])
        e['score_C'] = int(c_sc[i])
        e['score_D'] = int(d_sc[i])
        e['score_E'] = int(e_sc[i])

    # Print score distributions
    for mname, msc in [('A', a_sc), ('B', b_sc), ('C', c_sc), ('D', d_sc), ('E', e_sc)]:
        k_m = k if mname != 'E' else k_e
        dist = {s: int((msc == s).sum()) for s in range(1, k_m + 1)}
        l_ps = {s: round(float(X[msc == s, _L].mean()), 1)
                if (msc == s).any() else None for s in range(1, k_m + 1)}
        print(f'  {mname}: count={dist}')
        print(f'     L*  ={l_ps}')

    # Delta-E validation (primary method)
    print(f'\n  Delta-E CIEDE2000 within-cluster validation (threshold={DE_THRESHOLD}):')
    primary_sc = {'A': a_sc, 'B': b_sc, 'C': c_sc, 'D': d_sc, 'E': e_sc}[PRIMARY_METHOD]
    k_val = k if PRIMARY_METHOD != 'E' else k_e
    de_results = validate_cluster_deltaE(X, primary_sc, k_val)
    for s, r in de_results.items():
        status = 'OK' if r['ok'] else 'EXCEEDS'
        print(f'    Score {s}: n={r["n"]}  max_dE={r["max_dE"]:.2f}  '
              f'mean_dE={r["mean_dE"]:.2f}  [{status}]')

    # Save images under primary method scores
    for s in range(1, k_val + 1):
        folder = os.path.join(OUT_ROOT, color, f'Score={s:02d}')
        if os.path.exists(folder):
            shutil.rmtree(folder)
        os.makedirs(folder)
    for entry, sc in zip(ents, primary_sc):
        tiff.imwrite(
            os.path.join(OUT_ROOT, color, f'Score={int(sc):02d}', entry['fname']),
            entry['out'])
    print(f'  -> Saved {len(ents)} images to {OUT_ROOT}/{color}/ (method {PRIMARY_METHOD})')

print('\nAll methods computed and images saved.')

## Step 9 - Scoring Results & Visualisation

- **Score distributions** - counts per score for methods A/B/C/D (bar charts)
- **PCA 2D scatter** - images in reduced feature space, coloured by primary method score
- **Feature profiles** - L\* and a\* mean per score group (primary method)
- **Prediction frieze** - thumbnails sorted lightest to darkest, **5 rows (A/B/C/D/E)**
- **Method agreement** - 5x5 Kendall tau heatmap including Method E

In [ ]:
fig, axes = plt.subplots(len(COLORS), 3, figsize=(16, 5 * len(COLORS)))
fig.suptitle('Clustering Results - Score Distributions, PCA & Feature Profiles',
             fontsize=13, fontweight='bold')

for ri, color in enumerate(COLORS):
    res = method_results[color]
    k   = res['k']
    X   = feat_matrices[color]
    col = COLOR_MAP[color]

    # -- Column 1: score distribution per method ------------------
    ax = axes[ri, 0]
    x_ticks = np.arange(k)
    methods = ['A', 'B', 'C', 'D']
    width = 0.8 / len(methods)
    method_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
    for mi, (m, mc) in enumerate(zip(methods, method_colors)):
        counts = [(res[m] == s).sum() for s in range(1, k + 1)]
        ax.bar(x_ticks + mi * width - 0.3, counts, width,
               label=m, color=mc, alpha=0.75, edgecolor='white')
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f'S{s}' for s in range(1, k + 1)])
    ax.set_xlabel('Score'); ax.set_ylabel('Count')
    ax.set_title(f'{color} - Score Distribution (K={k})',
                 color=col, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

    # -- Column 2: PCA 2D scatter (primary method) ---------------
    ax = axes[ri, 1]
    pca = PCA(n_components=2, random_state=SEED)
    Xs  = StandardScaler().fit_transform(X)
    X_pca = pca.fit_transform(Xs)
    primary_sc = res[PRIMARY_METHOD]
    sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=primary_sc,
                    cmap='RdYlGn', s=120, edgecolors='white', linewidths=0.5,
                    vmin=1, vmax=k)
    for xp, yp, s in zip(X_pca[:, 0], X_pca[:, 1], primary_sc):
        ax.annotate(str(s), (xp, yp), fontsize=7, ha='center', va='center',
                    fontweight='bold')
    plt.colorbar(sc, ax=ax, label='Score')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'{color} - PCA (method {PRIMARY_METHOD})', fontsize=10)
    ax.grid(alpha=0.3)

    # -- Column 3: L* and a* mean per score ----------------------
    ax = axes[ri, 2]
    scores_range = range(1, k + 1)
    for feat, fi, lbl, lstyle in [
        ('L*_mean', _L, 'L* mean', 'o-'),
        ('a*_mean', _A, 'a* mean', 's--')]:
        vals_per_sc = [float(X[primary_sc == s, fi].mean()) if (primary_sc == s).any()
                       else np.nan for s in scores_range]
        ax.plot(list(scores_range), vals_per_sc, lstyle, lw=2, ms=8,
                label=lbl, alpha=0.85)
    ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlabel('Score'); ax.set_ylabel('Feature value')
    ax.set_title(f'{color} - L*/a* Profile per Score', fontsize=10)
    ax.set_xticks(list(scores_range)); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'clustering_overview.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Saved clustering_overview.png')

In [ ]:
# Prediction Frieze - one row per method, sorted light -> dark
THUMB      = 78
score_cmap = plt.cm.RdYlGn

for color in COLORS:
    entries = dataset[color]
    n_img   = len(entries)
    k       = K_MAP[color]
    k_e     = method_results[color]['k_e']
    src_col = COLOR_MAP[color]

    rows = [
        ('A  Formula', 'score_A', k),
        (f'B  K-Means (K={k})', 'score_B', k),
        (f'C  Hybrid  (K={k})', 'score_C', k),
        (f'D  a*Thresh (K={k})', 'score_D', k),
        (f'E  DeltaE (K={k_e})', 'score_E', k_e),
    ]
    n_rows = len(rows)

    fig, axes = plt.subplots(n_rows, n_img, figsize=(max(n_img * 1.3, 8), n_rows * 2.0))
    fig.suptitle(
        f'{color} - All Methods  (S1=lightest -> darkest)',
        fontsize=11, fontweight='bold', color=src_col)

    for row_i, (method_lbl, sk, k_row) in enumerate(rows):
        sorted_ents = sorted(entries, key=lambda e: (e[sk], e['fname']))
        prev_score  = None

        for col_i, ent in enumerate(sorted_ents):
            ax    = axes[row_i, col_i]
            score = ent[sk]
            clr   = score_cmap((score - 1) / max(k_row - 1, 1))
            ax.imshow(cv2.resize(ent['out'], (THUMB, THUMB)))
            ax.set_title(f'S{score}', fontsize=6, color=clr, fontweight='bold', pad=2)
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(True); sp.set_edgecolor(clr); sp.set_linewidth(2.5)
            if prev_score is not None and score != prev_score:
                ax.spines['left'].set_edgecolor('white')
                ax.spines['left'].set_linewidth(6)
            prev_score = score

        axes[row_i, 0].set_ylabel(
            method_lbl, fontsize=7, fontweight='bold',
            color=src_col, rotation=90, labelpad=4)

    plt.tight_layout(pad=0.35)
    safe = color.replace('Score=', '').lower()
    fname = f'frieze_{safe}.png'
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=140, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

## Step 10 - Export & Summary

- Images saved in structured folders: `<colour>/Score=XX/<image>.tif`
- Master CSV with all method scores, filter statistics, and metadata
- Final pipeline summary with per-colour statistics

In [ ]:
# Master CSV with all method scores (including Method E)
csv_rows = []
for color in COLORS:
    k = K_MAP[color]
    for entry in dataset[color]:
        row = {
            'color':       color.replace('Score=', ''),
            'fname':       entry['fname'],
            'kept_pct':    round(entry['kept_pct'], 2),
            'crop_side':   entry['crop_info']['side'],
            'score_A':     entry.get('score_A', ''),
            'score_B':     entry.get('score_B', ''),
            'score_C':     entry.get('score_C', ''),
            'score_D':     entry.get('score_D', ''),
            'score_E':     entry.get('score_E', ''),
            'primary':     entry.get(f'score_{PRIMARY_METHOD}', ''),
        }
        csv_rows.append(row)

df_labels = pd.DataFrame(csv_rows)
csv_path  = os.path.join(OUT_ROOT, 'lettuce_scores.csv')
df_labels.to_csv(csv_path, index=False)
print(f'CSV saved: {csv_path}')
print(df_labels.to_string(index=False))

# Summary
print(f'\n{"="*65}')
print(f'  LETTUCE COLOR SCORING - PIPELINE SUMMARY')
print(f'{"="*65}')
print(f'Dataset       : CSVS_Lettuce_202603_LaMenitre')
print(f'Input folder  : {LETTUCE_ROOT}')
print(f'Output folder : {OUT_ROOT}')
print(f'Filter        : K=+/-{K_STAT}s  (L*={FILTER_L} a*={FILTER_A} b*={FILTER_B})')
print(f'Features      : {len(feat_names_raw)} raw -> {len(feat_names)} after cleaning')
print(f'Corr threshold: {CORR_THRESHOLD}')
print(f'DE threshold  : {DE_THRESHOLD}  (Method E: {DE_THRESHOLD_E})')
print(f'Primary method: {PRIMARY_METHOD}')
print()
for color in COLORS:
    k   = K_MAP[color]
    res = method_results[color]
    n   = len(dataset[color])
    kp  = np.mean([e['kept_pct'] for e in dataset[color]])
    k_e = res['k_e']
    print(f'-- {color} ({n} images, K={k}, E:K={k_e}) -------')
    print(f'   Silhouette B={res["b_sil"]:.4f}  C={res["c_sil"]:.4f}')
    print(f'   Mean kept%: {kp:.1f}%')
    for mname in ['A', 'B', 'C', 'D', 'E']:
        k_m = k if mname != 'E' else k_e
        msc = res[mname]
        dist = {s: int((msc == s).sum()) for s in range(1, k_m + 1)}
        print(f'   {mname}: {dist}')
    print()

print('Output contents:')
for f in sorted(os.listdir(OUT_ROOT)):
    full = os.path.join(OUT_ROOT, f)
    if os.path.isdir(full):
        sub = os.listdir(full)
        print(f'  {f}/ ({len(sub)} items)')
    else:
        print(f'  {f}')
print(f'{"="*65}')